In [ ]:
import os
from crewai_tools import RagTool
from crewai.rag.config.utils import set_rag_config, get_rag_client, clear_rag_config
from crewai.rag.embeddings.factory import build_embedder
from crewai.rag.chromadb.config import ChromaDBConfig
from crewai.utilities.paths import db_storage_path
from com.example.ai.config import _rag_tool_config, create_rag_config, _embedder_config_st
from chromadb.config import Settings
from crewai.rag.config.types import RagConfigType

_rag_config = create_rag_config('openai')
embedding_function = build_embedder(_embedder_config_st)

#
set_rag_config(config=ChromaDBConfig(batch_size = 100, embedding_function=embedding_function))

#
chromadb_client = get_rag_client()

# Example operations (same API for any provider)
client = chromadb_client  # or chromadb_client



In [14]:
client.get_or_create_collection(collection_name="sandbox_documents")

Collection(name=sandbox_documents)

In [ ]:
import os 
from com.example.ai.loader.LoadManager import LoadManager
# 
#client.create_collection(collection_name="pdf-documents")

work_dir = os.getenv("WORK_DIR")
_pdf_documents = [
    f"{work_dir}/knowledge/pdf/Advanced_Microservices.pdf",
    # f"{work_dir}/knowledge/pdf/Building Microservices - Designing Fine-Grained Systems.pdf",
    # f"{work_dir}/knowledge/pdf/Clean Architecture A Craftsman Guide to Software Structure and Design.pdf",
    # f"{work_dir}/knowledge/pdf/Data Science.pdf",
    # f"{work_dir}/knowledge/pdf/Designing_Data_Intensive_Applications.pdf",
    # f"{work_dir}/knowledge/pdf/Enterprise Integration Patterns - Designing, Building And Deploying Messaging.pdf",
    # f"{work_dir}/knowledge/pdf/Kafka_Streams_In_Action.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices Best_Practices_for_Java.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices_From_Day_one.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices From Design to Deploy.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices Patterns.pdf",
    # f"{work_dir}/knowledge/pdf/ML_book.pdf",
    # f"{work_dir}/knowledge/pdf/Software Architecture Patterns.pdf",
    # f"{work_dir}/knowledge/pdf/System Design-interview-an-insiders.pdf"
]

import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len, separators=["\n\n", "\n", " ", ""])

for _pdf_doc in _pdf_documents :
    # Prepare data for ChromaDB
    _documents = LoadManager.from_pdfs([_pdf_doc])
    _documents_chunks = splitter.split_documents(_documents)
    if len(_documents_chunks):
        _docs_to_store = []
        for i,_d in enumerate(_documents_chunks) :
            _doc = dict()

            doc_id = f"pdf_{uuid.uuid4().hex[:8]}_{i}"
            _doc["id"] = doc_id
            _doc['content'] = _d.page_content

            # Prepare metadata
            _metadata = dict(_d.metadata)
            _metadata['doc_index'] = i
            _metadata['content_length'] = len(_d.page_content)
            #_metadata['content'] = _d.page_content
            _doc["metadata"] = _metadata
            
            # Document content
            _docs_to_store.append(_doc)
            
        # Add documents to vector store
        client.add_documents(
            collection_name="sandbox_documents",
            documents=_docs_to_store
        )
    else :
        print("document chunks not provided.")

In [16]:
results = client.search(collection_name="sandbox_documents", query="Advanced Microservices", limit=10)
for _result in results :
    print(_result)
    
#clear_rag_config()  # optional reset

In [17]:
results = client.search(collection_name="sandbox_documents", query="Kafaka Data Flow", limit=10)
for _result in results :
    print(_result)

#### RAG Tool Implementation

In [ ]:
from crewai_tools import RagTool
from com.example.ai.config import create_rag_config
# Create a RAG tool with custom configuration

_rag_tool_config = create_rag_config('openai')

rag_tool = RagTool(
    config=_rag_tool_config,
    collection_name="sandbox_documents", 
    summarize=True,
    limit=1000
)


In [ ]:
from crewai_tools.rag.data_types import DataType

work_dir = os.getenv("WORK_DIR")
_pdf_documents = [
    f"{work_dir}/knowledge/pdf/Advanced_Microservices.pdf",
    # f"{work_dir}/knowledge/pdf/Building Microservices - Designing Fine-Grained Systems.pdf",
    # f"{work_dir}/knowledge/pdf/Clean Architecture A Craftsman Guide to Software Structure and Design.pdf",
    # f"{work_dir}/knowledge/pdf/Data Science.pdf",
    # f"{work_dir}/knowledge/pdf/Designing_Data_Intensive_Applications.pdf",
    # f"{work_dir}/knowledge/pdf/Enterprise Integration Patterns - Designing, Building And Deploying Messaging.pdf",
    # f"{work_dir}/knowledge/pdf/Kafka_Streams_In_Action.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices Best_Practices_for_Java.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices_From_Day_one.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices From Design to Deploy.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices Patterns.pdf",
    # f"{work_dir}/knowledge/pdf/ML_book.pdf",
    #f"{work_dir}/knowledge/pdf/Software Architecture Patterns.pdf",
    #f"{work_dir}/knowledge/pdf/System Design-interview-an-insiders.pdf"
]
for pdf_file in _pdf_documents :
    # Add content from a file
    rag_tool.add(data_type=DataType.PDF_FILE, path=pdf_file)

# Add a txt file
#rag_tool.add(data_type=DataType.MDX, path=f"{work_dir}/knowledge/text/Brijesh_K_Dhaker.md")

# Add a web page
#rag_tool.add(data_type=DataType.WEBSITE, url="https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/rag/rag-solution-design-and-evaluation-guide")

# Add a YouTube video
#rag_tool.add(data_type=DataType.YOUTUBE_VIDEO, url="https://www.youtube.com/watch?v=VIDEO_ID")

# Add a directory of files
#rag_tool.add(data_type=DataType.DIRECTORY, path=f"{work_dir}/knowledge")

In [19]:
_rag_results = rag_tool.run(query="Microservice")
print(_rag_results)    

Relevant Content:
No relevant content found.


In [ ]:
from typing import Any

from crewai.knowledge.storage.base_knowledge_storage import BaseKnowledgeStorage
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource

from crewai.rag.types import SearchResult
from com.example.ai.config import _embedder_config_st
from pydantic import BaseModel, Field
from crewai.rag.embeddings.factory import build_embedder
from com.example.ai.vectors.ChromaVectorStore import ChromaVectorStore

# Create custom storage with specific embedder
custom_storage = KnowledgeStorage(
    collection_name="sandbox_documents",
    embedder=_embedder_config_st
)


class CustomChromaKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches data from Chromadb."""
    limit: int = Field(default=10, description="Number of articles to fetch")
    store: ChromaVectorStore = ChromaVectorStore()
    
    def load_content(self) -> list[SearchResult]:
        results = self.store.query("What are benefits of microservices ?", top_k=10)
        self.validate_content(results)
        return results
    
    def validate_content(self, articles: list) -> str:
        """Format ChromaDB document into readable text."""
        formatted = "Documents from Vector Store:\n"
        for article in articles:
            formatted += f"text: {article['content']}\nid: {article['id']}\n{'---'*50}\n"
        return formatted
    
    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        print(content)
        self._save_documents()

    def aadd(self) -> None:
        """Process and store the articles."""
        pass
#
kw_source = CustomChromaKnowledgeSource(
    limit=10,
    collection_name="sandbox_documents"
)

# #
test_query = ["test query"]
results = kw_source.load_content()
#kw_source.add()
print(f"knowledge results: {len(results)} documents found")



Loading embedding model: all-mpnet-base-v2
Model all-mpnet-base-v2 loaded successfully.
Embedding dimension: 768
Vector store initialized. Collection: sandbox_documents
Existing documents in collection: 0
[INFO] Querying vector store for: 'What are benefits of microservices ?'
Retrieving documents for query: 'What are benefits of microservices ?'
Top K: 10, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 768)
No documents found
knowledge results: 0 documents found


In [22]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
import requests
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field


class DesignKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches data from Vectorstore."""
    limit: int = Field(default=10, description="Number of articles to fetch")
    
    def load_content(self) -> Dict[Any, str]:
        """Fetch and format space news articles."""
        try:
            self._chunk_text("Hello")
            self._save_documents()
            
            self.storage.search()
            response = requests.get(
                f"{self.api_endpoint}?limit={self.limit}"
            )
            response.raise_for_status()

            data = response.json()
            articles = data.get('results', [])

            formatted_data = self.validate_content(articles)
            return {self.api_endpoint: formatted_data}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    def validate_content(self, articles: list) -> str:
        """Format articles into readable text."""
        formatted = "Space News Articles:\n\n"
        for article in articles:
            formatted += f"""
                Title: {article['title']}
                Published: {article['published_at']}
                Summary: {article['summary']}
                News Site: {article['news_site']}
                URL: {article['url']}
                -------------------"""
        return formatted

    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            self.chunks.extend(chunks)

        self._save_documents()

    def aadd(self) -> None:
        return super().aadd()        

In [ ]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from com.example.ai.config import _embedder_config_st

# Create custom storage with specific embedder
design_knowledge_storage = KnowledgeStorage(
    embedder=_embedder_config_st,
    collection_name="sandbox_documents"
)

# Create knowledge source
design_knowledge_source = DesignKnowledgeSource(
    collection_name="sandbox_documents",
    storage=design_knowledge_storage,
    limit=10,
    chunk_size=1000,
    chunk_overlap=200
)

In [ ]:
from crewai_tools import PDFSearchTool
from com.example.ai.config import create_rag_config

#pydantic.SkipValidation

# Create a RAG tool with custom configuration
_rag_tool_config = create_rag_config()

_pdf_search_tool = PDFSearchTool(
    config = _rag_tool_config,
    collection_name = "sandbox_documents", 
    summarize = True,
    limit = 1000,
    pdf=f"{os.getenv("WORK_DIR")}/knowledge/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf"
)


CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf


In [ ]:
work_dir = os.getenv("WORK_DIR")
_pdf_documents = [
    f"{work_dir}/knowledge/pdf/Advanced_Microservices.pdf",
    # f"{work_dir}/knowledge/pdf/Building Microservices - Designing Fine-Grained Systems.pdf",
    # f"{work_dir}/knowledge/pdf/Clean Architecture A Craftsman Guide to Software Structure and Design.pdf",
    # f"{work_dir}/knowledge/pdf/Data Science.pdf",
    # f"{work_dir}/knowledge/pdf/Designing_Data_Intensive_Applications.pdf",
    # f"{work_dir}/knowledge/pdf/Enterprise Integration Patterns - Designing, Building And Deploying Messaging.pdf",
    # f"{work_dir}/knowledge/pdf/Kafka_Streams_In_Action.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices Best_Practices_for_Java.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices_From_Day_one.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices From Design to Deploy.pdf",
    # f"{work_dir}/knowledge/pdf/Microservices Patterns.pdf",
    # f"{work_dir}/knowledge/pdf/ML_book.pdf",
    #f"{work_dir}/knowledge/pdf/Software Architecture Patterns.pdf",
    #f"{work_dir}/knowledge/pdf/System Design-interview-an-insiders.pdf"
]

for pdf_file in _pdf_documents :
    _pdf_search_tool.add(pdf=pdf_file)


In [31]:
results = _pdf_search_tool.run(query="JPMORGAN")
results

"Relevant Content:\n\n\n\n\nPage 2:\n\n \n\n• Designed and executed migration strategy from Cloudera Data Platform to Azure Databricks, leveraging Azure \n\nData Factory, ADX, and Apache Flink — reducing data processing latency by ~50%, cutting infrastructure \n\nlicensing costs by ~35%, and enabling elastic scaling for peak regulatory reporting windows without manual \n\nintervention. \n\n \n\n• Partnered with business, operations, and risk teams to identify and remediate 3 critical IT and operational risk \n\nitems (ORI, MERS, SERA), delivering solutions within agreed timelines and achieving full closure with zero \n\nregulatory escalations — strengthening UBS's control framework for IB Markets technology. \n\n \n\nVICE PRESIDENT – DATA & CLOUD TRANSFORMATION | JPMORGAN SERVICES | MUMBAI | MAY 2009 \n\n– MAR 2022 \n\n• Spearheaded a 13-year as Engineering Lead, multi-phase enterprise data platform (Investment Product & \n\nPortfolio Governance) serving Product & Portfolio Governance 

In [ ]:
from datetime import datetime
from crewai import Crew, Agent, Task, Process
from crewai import context
from emoji import config
from com.example.ai.config import _tool_config, _llm_openai

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(run_id)

from crewai_tools import FileReadTool

# Initialize tool
file_read_tool = FileReadTool(file_path=f"{os.environ["WORK_DIR"]}/knowledge/text/Brijesh_Resume.txt")
#file_read_tool.run()

# 3. Agent and Task Definition
text_specialist = Agent(
    role='Text File Reader',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[file_read_tool],
    #llm=llm
)

# 2. Define the Agent
resume_analyst = Agent(
    role='As a resume analyst',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[],
    allow_delegation=False,
    verbose=True,
    #llm=llm
)

# 3. Define the Task
read_task = Task(
    description='Read text content',
    expected_output='text contain in the file.',
    agent=text_specialist,
    output_file=f"{os.environ["WORK_DIR"]}/outputs/L004/read_task_{run_id}.txt"
)

extract_task = Task(
    description='review the content that you receive and make sure you extract skills, experience and work location',
    expected_output='A json string contain skills, experience and work location and list of domain knowledge',
    agent=text_specialist,
    context=[read_task],
    output_file=f"{os.environ["WORK_DIR"]}/outputs/L004/extract_task_{run_id}.json"
)

# 4. Run the Crew
crew = Crew(
    agents=[text_specialist], 
    tasks=[read_task,extract_task],
    process=Process.sequential,
    verbose=True,
    #planning=True,
    #planning_llm=_llm_openai
)
result = crew.kickoff()
print(result)

In [ ]:
def generate_paper_summary(text):

    try:

        llm = OpenAI(temperature=0.3, max_tokens=1000)  

        summary_prompt = (
            "You are an expert research analyst. Please provide a comprehensive summary of this research paper. "
            "Include the following sections:\n\n"
            "1. **Title and Authors** (if available)\n"
            "2. **Abstract/Summary** - Main research question and objectives\n"
            "3. **Methodology** - How the research was conducted\n"
            "4. **Key Findings** - Main results and discoveries\n"
            "5. **Contributions** - What new knowledge or insights this paper provides\n"
            "6. **Limitations** - Any limitations mentioned by the authors\n"
            "7. **Future Work** - Suggested future research directions\n\n"
            "Please be thorough but concise. Use clear headings and bullet points where appropriate.\n\n"
            f"Research Paper Text:\n{text[:8000]}\n\n"  # Limit text to avoid token limits
            "Summary:"
        )
        summary = llm.invoke(summary_prompt)
        return summary
    except Exception as e:
        st.error(f"Error generating summary: {str(e)}")
        return None

In [ ]:
import os
import shutil
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

def cleanup_chroma_db():
    try:
        if os.path.exists(f"{os.environ['WORK_DIR']}/storage/chromadb"):
            shutil.rmtree(f"{os.environ['WORK_DIR']}/storage/chromadb")
    except Exception as e:
        print(f"Exception - Could not clean up ChromaDB directory: {str(e)}")

def process_document(text):
    try:
        cleanup_chroma_db()
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100,
            length_function=len,
        )

        docs = splitter.create_documents([text])
        try:
            vectorstore = Chroma.from_documents(
                documents=docs,
                embedding=OpenAIEmbeddings(),
                collection_name="research_papers"
            )
            return vectorstore

        except Exception as chroma_error:
            print(f"WARNING - ChromaDB failed, trying FAISS: {str(chroma_error)}")
            vectorstore = FAISS.from_documents(
                documents=docs,
                embedding=OpenAIEmbeddings()
            )
            return vectorstore            

    except Exception as e:
        print(f"Exception - Error processing document: {str(e)}")
        cleanup_chroma_db()
        return None

In [ ]:
from openai import OpenAI

def retrieval_action(question, vectorstore):
    results = vectorstore.similarity_search(question, k=5)
    return "\n\n".join([f"Passage {i+1}: {doc.page_content}" for i, doc in enumerate(results)])

def generation_action(inputs):
    if isinstance(inputs, dict):
        question = inputs.get("user", "")
        context = inputs.get("Retriever", "")
    else:
        question = "User question not found in inputs"
        context = str(inputs)
    prompt = (
        "You are an expert research analyst. You have been given a specific research paper and asked a question about it. "
        "Based on the retrieved passages from the research paper, provide a detailed answer with numbered citations. "
        "Use only the information from the provided passages to answer the question.\n\n"
        f"USER QUESTION: {question}\n\n"
        f"RELEVANT PASSAGES FROM THE RESEARCH PAPER:\n{context}\n\n"
        "INSTRUCTIONS:\n"
        "1. Answer the question based ONLY on the provided passages from the research paper\n"
        "2. Use numbered citations [1], [2], etc. for each key point you reference\n"
        "3. If the passages don't contain enough information to answer, say so clearly\n"
        "4. Be specific and reference the actual content from the paper\n"
        "5. Focus on the key findings, results, and conclusions mentioned in the passages\n"
        "6. If asked about key findings, look for results, outcomes, discoveries, or conclusions\n\n"
        "ANSWER:"
    )    
    
    llm = OpenAI(
        model=os.environ['OPENAI_MODEL_NAME'],
        base_url='http://localhost:11434/v1/',
        api_key='ollama', # Required by SDK but ignored locally
        temperature=0.2, 
        max_tokens=1500
    )
    return llm.invoke(prompt)

def create_crew(vectorstore, status_callback=None):

    def retrieval_wrapper(question):
        if status_callback:
            status_callback("Searching for relevant passages in the research paper...")

        result = retrieval_action(question, vectorstore)
        if status_callback:
            status_callback("Found relevant passages for analysis")

        return result

    def generation_wrapper(inputs):
        if status_callback:
            status_callback("Analyzing retrieved passages and generating comprehensive answer...")

        result = generation_action(inputs)
        if status_callback:
            status_callback("Generated detailed answer with citations")

        return result

    retriever = Agent(
        role="Retriever",
        goal="Retrieve relevant passages from the research paper",
        backstory="You are an expert at finding relevant information in research papers",
        action=retrieval_wrapper,
        verbose=True
    )

    generator = Agent(
        role="Generator", 
        goal="Generate comprehensive answers based on retrieved passages",
        backstory="You are an expert research analyst who provides detailed, citation-based answers",
        action=generation_wrapper,
        verbose=True
    )

    retrieval_task = Task(
        description="Retrieve relevant passages from the research paper for the user's question",
        agent=retriever,
        expected_output="Relevant passages from the research paper that can answer the user's question"
    )

    generation_task = Task(
        description="Generate a comprehensive answer with citations based on the retrieved passages from the research paper",
        agent=generator,
        expected_output="A detailed answer with numbered citations based on the research paper content",
        context=[retrieval_task]
    )    

    crew = Crew(
        agents=[retriever, generator],
        tasks=[retrieval_task, generation_task],
        verbose=True
    )

    return crew

In [ ]:
def main():

    st.title("Research Paper Analyst")
    st.markdown("Upload a research paper and ask questions about it using AI-powered analysis.")    
    with st.sidebar:
        st.header("Document Upload")
        uploaded_file = st.file_uploader(
            "Choose a PDF file",
            type="pdf",
            help="Upload a research paper in PDF format"
        )        

        if uploaded_file is not None:
            st.info(f"File uploaded: {uploaded_file.name}")        
            if st.button("Process Document", type="primary"):
                with st.spinner("Processing document and generating summary..."):
                    text = extract_text_from_pdf(uploaded_file)    
                    if text:
                        summary = generate_paper_summary(text)        
                        if summary:
                            st.session_state.paper_summary = summary
                            vectorstore = process_document(text)        
                        if vectorstore:
                            st.session_state.vectorstore = vectorstore
                            st.session_state.document_processed = True
                            st.info("Document processed successfully! Summary generated and ready for questions.")
                        else:
                            st.error("Failed to process document.")
                    else:
                        st.error("Failed to extract text from PDF.")

    if not st.session_state.document_processed:
        st.info("Please upload and process a PDF document using the sidebar to get started.")   

    else:
        with st.expander("Paper Summary", expanded=False):
            if st.session_state.paper_summary:
                st.markdown(st.session_state.paper_summary)
            else:
                st.warning("Summary not available.")

        st.subheader("Ask Questions About Your Research Paper")        

        for message in st.session_state.chat_history:
            with st.chat_message(message["role"]):
                st.write(message["content"]) 

        if prompt := st.chat_input("Ask a question about the research paper..."):
            st.session_state.chat_history.append({"role": "user", "content": prompt})  

            with st.chat_message("user"):
                st.write(prompt)        

            with st.chat_message("assistant"):
                progress_container = st.container()
                execution_trace_container = st.expander("Execution Details", expanded=False)     

                with progress_container:
                    st.info("Initializing AI agents...")

                try:
                    status_placeholder = st.empty()
                    trace_placeholder = execution_trace_container.empty()
                    execution_steps = []

                    def log_step(step):
                        execution_steps.append(step)
                        with trace_placeholder:
                            for i, s in enumerate(execution_steps, 1):
                                st.write(f"{i}. {s}")

                    def status_callback(message):
                        status_placeholder.info(message)
                        log_step(message)                    

                    status_placeholder.info("Initializing AI agents...")
                    log_step("CrewAI agents initialized")
                    crew = create_crew(st.session_state.vectorstore, status_callback)
                    status_placeholder.info("Starting AI analysis...")
                    log_step("CrewAI execution started")
                    status_placeholder.info("Retrieving relevant passages...")
                    retrieved_passages = retrieval_action(prompt, st.session_state.vectorstore)
                    log_step("Retrieved relevant passages from the research paper")
                    status_placeholder.info("Generating comprehensive answer...")

                    inputs = {
                        "user": prompt,
                        "Retriever": retrieved_passages
                    }

                    response = generation_action(inputs)
                    log_step("Generated detailed answer with citations")
                    status_placeholder.info("Analysis complete!")
                    st.session_state.chat_history.append({"role": "assistant", "content": response})
                    st.markdown("### AI Analysis Result:")
                    st.write(response)
                    with execution_trace_container:
                        st.markdown("### Execution Summary:")
                        st.info("**Retriever Agent**: Found relevant passages from the research paper")
                        st.info("**Generator Agent**: Created comprehensive answer with citations")
                        st.info(f"**Total Steps**: {len(execution_steps)}")
                        st.markdown("### Detailed Execution Trace:")

                        for i, step in enumerate(execution_steps, 1):
                            st.write(f"{i}. {step}")            

                except Exception as e:
                    error_msg = f"Error generating response: {str(e)}"
                    st.error(error_msg)
                    st.session_state.chat_history.append({"role": "assistant", "content": error_msg})

        

        if st.button("Clear Chat History"):
            st.session_state.chat_history = []
            st.rerun()


‍if __name__ == "__main__":
    main()